# AksharaPiece: publish to Hugging Face

Uploads everything from the project so anyone can use it:

| Hugging Face repository | Contents |
|---|---|
| **`<you>/aksharapiece-nepali`** (model) | 2 tokenizers, the 8 trained models (as safetensors), loading code, `example.py`, and a model card with the results |
| **`<you>/aksharapiece-nepali-eval`** (dataset) | 48,000 generated continuations (free and script-aware), all metrics, per-line language-model scores |

**Before you run**

1. **Make a Write token** on huggingface.co: profile picture → **Settings → Access Tokens → Create new token → Write**. Copy it once.
2. In this notebook: **Add-ons → Secrets → Add a new secret**, name **`HF_WRITE_TOKEN`**, paste the token, and switch it on for this notebook. Never paste the token into a cell.
3. **Add Input → Notebooks** → add **aksharapiece_seeds** (the models) and **aksharapiece_eval** (its latest version holds the generation results).
4. **Accelerator: None** (no GPU needed). **Internet: On**.
5. Run the cells one by one, or **Run All**. It takes about 5 to 10 minutes, mostly uploading about 600 MB.

The repositories are created **public**. Set `PRIVATE = True` below if you want to check them first and make them public later on the website.

In [ ]:
# ---- Settings ----
MODELS_REPO = "aksharapiece-nepali"        # becomes https://huggingface.co/<you>/aksharapiece-nepali
DATA_REPO = "aksharapiece-nepali-eval"     # becomes https://huggingface.co/datasets/<you>/aksharapiece-nepali-eval
GITHUB_URL = ""                            # e.g. "https://github.com/<you>/aksharapiece" once the GitHub repo exists
PRIVATE = False
UPLOAD = True                              # False = only build and show the files, upload nothing

## Setup
Unpacks the project code and finds the trained models and results from your other notebooks.

In [ ]:
import base64, glob, io, os, subprocess, sys, zipfile

ON_KAGGLE = os.path.isdir("/kaggle/working")
INPUT = "/kaggle/input" if ON_KAGGLE else os.environ.get("RELEASE_INPUT", os.getcwd())
WORK = "/tmp/aksharapiece_release" if ON_KAGGLE else os.path.join(os.getcwd(), "work")
os.makedirs(WORK, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode("..."))).extractall(WORK)
PROJECT = os.path.join(WORK, "aksharapiece")
os.chdir(PROJECT)


def find(pattern):
    return sorted(glob.glob(os.path.join(INPUT, "**", pattern), recursive=True))


MODELS = find("models/*.pt")
LM_ZIP = (find("aksharapiece_lm_results.zip") or [None])[0]
GEN_ZIP = (find("aksharapiece_gen_results.zip") or [None])[0]
print(len(MODELS), "models:", *[os.path.basename(m) for m in MODELS], sep="\n  ")
print("LM results zip:", LM_ZIP)
print("generation results zip:", GEN_ZIP or "not found (add aksharapiece_eval as input; samples will be skipped)")
if len(MODELS) == 0 or LM_ZIP is None:
    raise RuntimeError("Add Input -> Notebooks -> aksharapiece_seeds (needs models/ and aksharapiece_lm_results.zip).")

In [ ]:
TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("HF_WRITE_TOKEN")
except Exception:
    TOKEN = os.environ.get("HF_WRITE_TOKEN")
if UPLOAD and not TOKEN:
    raise RuntimeError("No HF_WRITE_TOKEN secret. Add-ons -> Secrets -> add HF_WRITE_TOKEN (a Write token) and switch it on.")
USER = "USERNAME"
if TOKEN:
    from huggingface_hub import HfApi
    info = HfApi(token=TOKEN).whoami()
    USER = info["name"]
    role = (info.get("auth", {}).get("accessToken", {}) or {}).get("role")
    print("Logged in as", USER, "| token type:", role or "unknown")
    if role == "read":
        raise RuntimeError("This is a Read token. Make a Write token for uploading.")

## Build and upload
Converts each model to safetensors, writes the model and dataset cards, then uploads both repositories.

In [ ]:
env = dict(os.environ)
if TOKEN:
    env["HF_TOKEN"] = TOKEN                 # passed to the script only; never printed
cmd = [sys.executable, "scripts/hf_release.py", "--models-dir", os.path.dirname(MODELS[0]), "--lm-zip", LM_ZIP,
       "--out", os.path.join(WORK, "release"), "--user", USER, "--models-repo", MODELS_REPO, "--data-repo", DATA_REPO,
       "--github-url", GITHUB_URL]
if GEN_ZIP:
    cmd += ["--gen-zip", GEN_ZIP]
if PRIVATE:
    cmd.append("--private")
if UPLOAD:
    cmd.append("--upload")
p = subprocess.run(cmd, env=env, capture_output=True, text=True)
print(p.stdout[-6000:])
if p.returncode != 0:
    print(p.stderr[-4000:].replace(TOKEN or "@@", "***"))
    raise RuntimeError("release failed (see above)")

## Check it works
Downloads the model repository again and generates a sentence with an uploaded model, the way any user would.

In [ ]:
if UPLOAD:
    from huggingface_hub import snapshot_download
    root = snapshot_download(f"{USER}/{MODELS_REPO}", token=TOKEN)
else:
    root = os.path.join(WORK, "release", "models_repo")
sys.path.insert(0, os.path.join(root, "code"))
from example import complete, load

name = sorted(os.listdir(os.path.join(root, "models")))[0]
model, codec = load(name, root=root)
for constrained in (False, True):
    print(f"{name} | script-aware={constrained}:", complete(model, codec, ["नेपालको राजधानी"], max_new=30, constrained=constrained)[0])
print()
print("Model repo:  ", f"https://huggingface.co/{USER}/{MODELS_REPO}")
print("Dataset repo:", f"https://huggingface.co/datasets/{USER}/{DATA_REPO}")

## After uploading

- Open both links above and check the model card renders.
- In the paper (`main.tex`), replace `USERNAME` in the footnote with your Hugging Face user name.
- When the GitHub repository is up, set `GITHUB_URL` above and run again: it updates the cards (uploading again only changes what differs).
- You can delete the `HF_WRITE_TOKEN` secret afterwards, or revoke the token on Hugging Face.